# Monitorando a Ciência Aberta
## Geração de indicadores com dados do OpenAlex

**Workshop — ConfOA 2026** · duração 3h

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiocantoadv/OA_monitor/blob/main/notebooks/OA_Monitor_Workshop_ConfOA2026.ipynb)

Fabio Lorensi do Canto (UFSC/IBICT) · Thiago M. R. Dias (CEFET-MG/IBICT) · Marcel Garcia de Souza (IBICT) · Washington L. R. Carvalho Segundo (IBICT)

---

### O que você vai fazer aqui

| Bloco | O quê |
|---|---|
| **0** | Configurar sua credencial individual do OpenAlex |
| **2.1** | Escolher um recorte (país, instituição por ROR, pesquisador por ORCID) e extrair os dados |
| **2.2** | Calcular indicadores: taxa de abertura, vias, APCs, citação |
| **2.3** | Visualizar os resultados |
| **3** | Gerar um dashboard interativo (Vega) e baixá-lo |

**Como rodar:** `Ambiente de execução → Executar tudo` (ou `Ctrl+F9`), ou célula a célula com `Shift+Enter`.

> Todo o código deste notebook também está em [`https://github.com/fabiocantoadv/OA_monitor`](https://github.com/fabiocantoadv/OA_monitor) como um pacote Python reutilizável (`src/oa_monitor/`).

---
# Parte 0 — Preparação

## 0.1 Instalação

O Colab já traz `pandas`, `matplotlib` e `requests`. Esta célula só garante as versões.

In [ ]:
%pip install -q requests pandas matplotlib

import json, os, re, time, html
from dataclasses import dataclass
from datetime import datetime
from typing import Any, Callable, Iterable, Iterator

import matplotlib.pyplot as plt
import pandas as pd
import requests
from matplotlib.ticker import PercentFormatter

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
print('Ambiente pronto.')


---
## 0.2 Código de apoio

As quatro células a seguir definem tudo o que o notebook usa. **Leia-as** — é este o código que você vai adaptar ao seu contexto depois (inclusive pedindo a um LLM: *"altere a função `montar_filtro` para aceitar também financiador"*).

É o mesmo código do pacote `oa_monitor` no repositório.

### Credenciais e cliente HTTP

O cliente cuida de três coisas chatas: **repetir** requisições que falham (com espera crescente), **paginar** por cursor (sem o teto de 10 mil registros da paginação por página) e **ler os cabeçalhos de cota** que o OpenAlex devolve.

In [ ]:
import os
from dataclasses import dataclass

OPENALEX_BASE_URL = "https://api.openalex.org"

# Versão do material, enviada no User-Agent para facilitar o suporte.
USER_AGENT_APP = "OA_monitor/1.0 (+https://github.com/fabiocantoadv/OA_monitor)"


@dataclass
class Credentials:
    """Credenciais individuais do participante."""

    mailto: str
    api_key: str | None = None

    def __post_init__(self) -> None:
        self.mailto = (self.mailto or "").strip()
        self.api_key = (self.api_key or "").strip() or None
        if "@" not in self.mailto or "." not in self.mailto.split("@")[-1]:
            raise ValueError(
                "Informe um e-mail válido em `mailto`. Ele identifica você no "
                "polite pool do OpenAlex e é obrigatório neste material."
            )

    @classmethod
    def from_env(cls) -> "Credentials":
        """Lê OPENALEX_MAILTO e OPENALEX_API_KEY do ambiente."""
        return cls(
            mailto=os.environ.get("OPENALEX_MAILTO", ""),
            api_key=os.environ.get("OPENALEX_API_KEY"),
        )

    @property
    def auth_params(self) -> dict[str, str]:
        params = {"mailto": self.mailto}
        if self.api_key:
            params["api_key"] = self.api_key
        return params

    @property
    def user_agent(self) -> str:
        return f"{USER_AGENT_APP} mailto:{self.mailto}"

    def describe(self) -> str:
        chave = "sim (cota individual)" if self.api_key else "não (cota compartilhada pelo IP)"
        return f"mailto={self.mailto} | api_key={chave}"


import time
from typing import Any, Callable, Iterable, Iterator

import requests


MAX_PER_PAGE = 200
DEFAULT_TIMEOUT = 60
RETRY_STATUS = {429, 500, 502, 503, 504}


class OpenAlexError(RuntimeError):
    pass


class OpenAlexClient:
    """Cliente com retry, paginação por cursor e leitura de cota.

    >>> cli = OpenAlexClient(Credentials(mailto="voce@exemplo.br"))
    >>> cli.count("works", {"filter": "publication_year:2024"})
    """

    def __init__(
        self,
        credentials: Credentials,
        base_url: str = OPENALEX_BASE_URL,
        max_retries: int = 5,
        pause: float = 0.1,
    ) -> None:
        self.credentials = credentials
        self.base_url = base_url.rstrip("/")
        self.max_retries = max_retries
        self.pause = pause
        self.session = requests.Session()
        self.session.headers.update({"User-Agent": credentials.user_agent})
        self.last_rate_limit: dict[str, str] = {}

    # ------------------------------------------------------------------ #
    # requisição bruta
    # ------------------------------------------------------------------ #
    def get(self, endpoint: str, params: dict[str, Any] | None = None) -> dict:
        url = f"{self.base_url}/{endpoint.lstrip('/')}"
        query = {**(params or {}), **self.credentials.auth_params}
        query = {k: v for k, v in query.items() if v not in (None, "")}

        delay = 1.0
        last_error = ""
        for tentativa in range(1, self.max_retries + 1):
            try:
                resp = self.session.get(url, params=query, timeout=DEFAULT_TIMEOUT)
            except requests.RequestException as exc:  # rede instável
                last_error = f"falha de rede: {exc}"
            else:
                self._store_rate_limit(resp)
                if resp.status_code == 200:
                    if self.pause:
                        time.sleep(self.pause)
                    return resp.json()
                if resp.status_code == 403:
                    raise OpenAlexError(
                        "403 do OpenAlex. Verifique sua api_key / mailto na célula "
                        f"de configuração. Resposta: {resp.text[:300]}"
                    )
                if resp.status_code not in RETRY_STATUS:
                    raise OpenAlexError(f"HTTP {resp.status_code}: {resp.text[:300]}")
                last_error = f"HTTP {resp.status_code}"

            if tentativa < self.max_retries:
                time.sleep(delay)
                delay = min(delay * 2, 30)

        raise OpenAlexError(
            f"Não foi possível completar a requisição após {self.max_retries} "
            f"tentativas ({last_error}). URL: {url}"
        )

    def _store_rate_limit(self, resp: requests.Response) -> None:
        for header in (
            "X-RateLimit-Limit",
            "X-RateLimit-Remaining",
            "X-RateLimit-Credits-Used",
            "X-RateLimit-Reset",
        ):
            if header in resp.headers:
                self.last_rate_limit[header] = resp.headers[header]

    # ------------------------------------------------------------------ #
    # helpers de alto nível
    # ------------------------------------------------------------------ #
    def count(self, endpoint: str, params: dict[str, Any] | None = None) -> int:
        """Quantos registros existem para o filtro, sem baixar os registros."""
        payload = self.get(endpoint, {**(params or {}), "per-page": 1})
        return int(payload.get("meta", {}).get("count", 0))

    def group_by(
        self, endpoint: str, group: str, params: dict[str, Any] | None = None
    ) -> list[dict]:
        """Agregação server-side: 1 requisição em vez de baixar tudo.

        Retorna a lista ``[{key, key_display_name, count}, ...]``.
        """
        payload = self.get(endpoint, {**(params or {}), "group_by": group, "per-page": 200})
        return payload.get("group_by", [])

    def paginate(
        self,
        endpoint: str,
        params: dict[str, Any] | None = None,
        max_records: int | None = None,
        per_page: int = MAX_PER_PAGE,
        on_progress: Callable[[int, int], None] | None = None,
    ) -> Iterator[dict]:
        """Percorre resultados usando paginação por cursor (sem limite de 10 mil)."""
        cursor = "*"
        baixados = 0
        total = None
        while cursor:
            payload = self.get(
                endpoint,
                {**(params or {}), "per-page": min(per_page, MAX_PER_PAGE), "cursor": cursor},
            )
            meta = payload.get("meta", {})
            if total is None:
                total = int(meta.get("count", 0))
                if max_records:
                    total = min(total, max_records)
            results = payload.get("results", [])
            if not results:
                break
            for item in results:
                yield item
                baixados += 1
                if max_records and baixados >= max_records:
                    if on_progress:
                        on_progress(baixados, total or baixados)
                    return
            if on_progress:
                on_progress(baixados, total or baixados)
            cursor = meta.get("next_cursor")

    def rate_limit_status(self) -> dict:
        """Consulta o endpoint /rate-limit (exige api_key)."""
        if not self.credentials.api_key:
            return {"detalhe": "sem api_key: cota compartilhada pelo IP da rede"}
        return self.get("rate-limit")


def chunked(itens: Iterable[str], tamanho: int = 50) -> Iterator[list[str]]:
    """Divide uma lista de IDs em blocos para o filtro ``id:a|b|c`` do OpenAlex."""
    bloco: list[str] = []
    for item in itens:
        bloco.append(item)
        if len(bloco) == tamanho:
            yield bloco
            bloco = []
    if bloco:
        yield bloco


### Filtros e extração

`montar_filtro` traduz *"a UFSC entre 2015 e 2025"* para a sintaxe do parâmetro `filter` da API. `achatar_work` transforma o JSON aninhado do OpenAlex numa linha de tabela — é aqui que você acrescenta um campo, se precisar de outro.

In [ ]:
import re
from typing import Any, Callable

import pandas as pd


# Campos pedidos ao OpenAlex. Pedir só o necessário deixa a resposta muito
# menor e a extração bem mais rápida.
WORK_FIELDS = [
    "id",
    "doi",
    "title",
    "display_name",
    "publication_year",
    "publication_date",
    "type",
    "language",
    "cited_by_count",
    "fwci",
    "is_retracted",
    "open_access",
    "apc_list",
    "apc_paid",
    "primary_location",
    "best_oa_location",
    "locations_count",
    "authorships",
    "primary_topic",
]

OA_STATUS_ORDEM = ["diamond", "hybrid", "gold", "green", "bronze", "closed"]

OA_STATUS_ROTULO = {
    "diamond": "Diamante",
    "hybrid": "Híbrido",
    "gold": "Dourado",
    "green": "Verde",
    "bronze": "Bronze",
    "closed": "Fechado",
}


# --------------------------------------------------------------------------- #
# normalização de identificadores
# --------------------------------------------------------------------------- #
def normalizar_ror(valor: str) -> str:
    """Aceita ``https://ror.org/041akq887``, ``ror.org/041akq887`` ou ``041akq887``."""
    valor = (valor or "").strip()
    m = re.search(r"(0[0-9a-hj-km-np-tv-z]{6}[0-9]{2})", valor, flags=re.I)
    if not m:
        raise ValueError(
            f"ROR inválido: {valor!r}. Procure o ROR da instituição em "
            "https://ror.org (ex.: 041akq887 para a UFSC)."
        )
    return f"https://ror.org/{m.group(1).lower()}"


def normalizar_orcid(valor: str) -> str:
    valor = (valor or "").strip()
    m = re.search(r"(\d{4}-\d{4}-\d{4}-\d{3}[\dXx])", valor)
    if not m:
        raise ValueError(f"ORCID inválido: {valor!r}. Formato esperado: 0000-0002-8338-1931.")
    return f"https://orcid.org/{m.group(1).upper()}"


def normalizar_pais(valor: str) -> str:
    valor = (valor or "").strip().upper()
    if not re.fullmatch(r"[A-Z]{2}", valor):
        raise ValueError(f"Código de país inválido: {valor!r}. Use ISO 3166-1 alfa-2, ex.: BR, PT.")
    return valor


# --------------------------------------------------------------------------- #
# montagem do filtro
# --------------------------------------------------------------------------- #
def montar_filtro(
    nivel: str,
    identificador: str,
    ano_inicio: int,
    ano_fim: int,
    tipos: list[str] | None = None,
    somente_com_doi: bool = False,
    extras: dict[str, str] | None = None,
) -> str:
    """Monta a string do parâmetro ``filter`` da API.

    nivel: ``"pais"``, ``"instituicao"``, ``"pesquisador"`` ou ``"fonte"``.
    identificador: código de país (BR), ROR, ORCID ou ISSN, conforme o nível.
    """
    nivel = nivel.strip().lower()
    if nivel in ("pais", "país", "country"):
        chave = f"authorships.institutions.country_code:{normalizar_pais(identificador)}"
    elif nivel in ("instituicao", "instituição", "institution", "ror"):
        chave = f"authorships.institutions.ror:{normalizar_ror(identificador)}"
    elif nivel in ("pesquisador", "autor", "author", "orcid"):
        chave = f"authorships.author.orcid:{normalizar_orcid(identificador)}"
    elif nivel in ("fonte", "source", "issn", "periodico", "periódico"):
        chave = f"primary_location.source.issn:{identificador.strip()}"
    else:
        raise ValueError(
            f"Nível desconhecido: {nivel!r}. Use pais, instituicao, pesquisador ou fonte."
        )

    partes = [chave, f"publication_year:{ano_inicio}-{ano_fim}"]
    if tipos:
        partes.append("type:" + "|".join(tipos))
    if somente_com_doi:
        partes.append("has_doi:true")
    for k, v in (extras or {}).items():
        partes.append(f"{k}:{v}")
    return ",".join(partes)


# --------------------------------------------------------------------------- #
# extração
# --------------------------------------------------------------------------- #
def _autor_instituicoes(work: dict) -> tuple[str, str, int]:
    nomes, paises = [], []
    for a in work.get("authorships") or []:
        for inst in a.get("institutions") or []:
            if inst.get("display_name"):
                nomes.append(inst["display_name"])
            if inst.get("country_code"):
                paises.append(inst["country_code"])
    n_autores = len(work.get("authorships") or [])
    return ("; ".join(dict.fromkeys(nomes))[:500], "; ".join(dict.fromkeys(paises)), n_autores)


def achatar_work(work: dict) -> dict:
    """Converte um registro aninhado do OpenAlex numa linha de tabela."""
    oa = work.get("open_access") or {}
    pl = work.get("primary_location") or {}
    fonte = pl.get("source") or {}
    boa = work.get("best_oa_location") or {}
    apc_list = work.get("apc_list") or {}
    apc_paid = work.get("apc_paid") or {}
    topico = work.get("primary_topic") or {}
    instituicoes, paises, n_autores = _autor_instituicoes(work)

    return {
        "id": work.get("id"),
        "doi": work.get("doi"),
        "titulo": work.get("title") or work.get("display_name"),
        "ano": work.get("publication_year"),
        "data": work.get("publication_date"),
        "tipo": work.get("type"),
        "idioma": work.get("language"),
        "citacoes": work.get("cited_by_count", 0),
        "fwci": work.get("fwci"),
        "retratado": work.get("is_retracted", False),
        # --- acesso aberto ---
        "is_oa": bool(oa.get("is_oa", False)),
        "oa_status": oa.get("oa_status", "closed"),
        "oa_url": oa.get("oa_url"),
        "em_repositorio": bool(oa.get("any_repository_has_fulltext", False)),
        "melhor_oa_tipo": boa.get("version"),
        "melhor_oa_local": ((boa.get("source") or {}).get("type")),
        # --- fonte ---
        "fonte": fonte.get("display_name"),
        "fonte_id": fonte.get("id"),
        "fonte_issn_l": fonte.get("issn_l"),
        "fonte_tipo": fonte.get("type"),
        "fonte_editora": fonte.get("host_organization_name"),
        "fonte_no_doaj": bool(fonte.get("is_in_doaj", False)),
        "fonte_is_oa": bool(fonte.get("is_oa", False)),
        # --- APC ---
        "apc_cobrado_usd": apc_list.get("value_usd"),
        "apc_pago_usd": apc_paid.get("value_usd"),
        "apc_pago_origem": apc_paid.get("provenance"),
        # --- contexto ---
        "topico": topico.get("display_name"),
        "area": ((topico.get("field") or {}).get("display_name")),
        "instituicoes": instituicoes,
        "paises_instituicoes": paises,
        "n_autores": n_autores,
    }


def extrair_works(
    client: OpenAlexClient,
    filtro: str,
    max_registros: int | None = 5000,
    on_progress: Callable[[int, int], None] | None = None,
) -> pd.DataFrame:
    """Baixa as obras que atendem ao filtro e devolve um DataFrame achatado."""
    params: dict[str, Any] = {"filter": filtro, "select": ",".join(WORK_FIELDS)}
    linhas = [
        achatar_work(w)
        for w in client.paginate(
            "works", params, max_records=max_registros, on_progress=on_progress
        )
    ]
    df = pd.DataFrame(linhas)
    if df.empty:
        return df

    df["oa_status"] = (
        df["oa_status"].fillna("closed").where(df["oa_status"].isin(OA_STATUS_ORDEM), "closed")
    )
    df["oa_status_rotulo"] = df["oa_status"].map(OA_STATUS_ROTULO)
    df["ano"] = pd.to_numeric(df["ano"], errors="coerce").astype("Int64")
    for col in ("apc_cobrado_usd", "apc_pago_usd", "fwci"):
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def resumo_rapido(client: OpenAlexClient, filtro: str) -> dict:
    """Contagens agregadas sem baixar registro nenhum (1 requisição cada).

    Útil para dimensionar a extração antes de rodá-la.
    """
    total = client.count("works", {"filter": filtro})
    por_status = {
        g["key"]: g["count"] for g in client.group_by("works", "open_access.oa_status", {"filter": filtro})
    }
    return {"total": total, "por_oa_status": por_status}


### Indicadores

Cada função recebe o DataFrame e devolve outro DataFrame, pronto para gráfico ou para exportar. Nada de estado escondido.

In [ ]:
import pandas as pd



def _vazio(df: pd.DataFrame) -> bool:
    return df is None or len(df) == 0


# --------------------------------------------------------------------------- #
# 1. Indicadores de cabeçalho
# --------------------------------------------------------------------------- #
def indicadores_gerais(df: pd.DataFrame) -> dict:
    """Números-síntese do conjunto analisado."""
    if _vazio(df):
        return {}
    n = len(df)
    oa = int(df["is_oa"].sum())
    com_apc = df["apc_pago_usd"].notna() & (df["apc_pago_usd"] > 0)
    return {
        "obras": n,
        "periodo": f"{int(df['ano'].min())}–{int(df['ano'].max())}",
        "obras_oa": oa,
        "taxa_oa": round(100 * oa / n, 1),
        "taxa_diamante": round(100 * (df["oa_status"] == "diamond").sum() / n, 1),
        "taxa_em_repositorio": round(100 * df["em_repositorio"].sum() / n, 1),
        "taxa_no_doaj": round(100 * df["fonte_no_doaj"].sum() / n, 1),
        "obras_com_apc_pago": int(com_apc.sum()),
        "apc_total_usd": round(float(df.loc[com_apc, "apc_pago_usd"].sum()), 2),
        "apc_medio_usd": (
            round(float(df.loc[com_apc, "apc_pago_usd"].mean()), 2) if com_apc.any() else 0.0
        ),
        "citacoes_totais": int(df["citacoes"].sum()),
        "citacoes_por_obra": round(float(df["citacoes"].mean()), 2),
    }


# --------------------------------------------------------------------------- #
# 2. Distribuição por via de acesso
# --------------------------------------------------------------------------- #
def distribuicao_oa(df: pd.DataFrame) -> pd.DataFrame:
    """Obras por via de acesso (diamante, híbrido, dourado, verde, bronze, fechado)."""
    if _vazio(df):
        return pd.DataFrame(columns=["oa_status", "rotulo", "obras", "percentual"])
    cont = df["oa_status"].value_counts()
    linhas = [
        {
            "oa_status": s,
            "rotulo": OA_STATUS_ROTULO[s],
            "obras": int(cont.get(s, 0)),
            "percentual": round(100 * int(cont.get(s, 0)) / len(df), 2),
        }
        for s in OA_STATUS_ORDEM
    ]
    return pd.DataFrame(linhas)


# --------------------------------------------------------------------------- #
# 3. Séries temporais
# --------------------------------------------------------------------------- #
def serie_anual_oa(df: pd.DataFrame) -> pd.DataFrame:
    """Taxa de acesso aberto por ano."""
    if _vazio(df):
        return pd.DataFrame(columns=["ano", "obras", "obras_oa", "taxa_oa"])
    g = df.groupby("ano", dropna=True).agg(obras=("id", "size"), obras_oa=("is_oa", "sum"))
    g = g.reset_index()
    g["taxa_oa"] = (100 * g["obras_oa"] / g["obras"]).round(2)
    g["ano"] = g["ano"].astype(int)
    return g.sort_values("ano").reset_index(drop=True)


def serie_anual_por_status(df: pd.DataFrame, percentual: bool = True) -> pd.DataFrame:
    """Composição das vias de acesso ano a ano (formato longo, pronto para Vega)."""
    if _vazio(df):
        return pd.DataFrame(columns=["ano", "oa_status", "rotulo", "obras", "percentual", "ordem"])
    g = (
        df.groupby(["ano", "oa_status"], dropna=True)
        .size()
        .reset_index(name="obras")
    )
    total = g.groupby("ano")["obras"].transform("sum")
    g["percentual"] = (100 * g["obras"] / total).round(2)
    g["rotulo"] = g["oa_status"].map(OA_STATUS_ROTULO)
    g["ano"] = g["ano"].astype(int)
    # `ordem` fixa a sequência semântica das vias (diamante → fechado). É ela que
    # empilha os segmentos na mesma ordem no matplotlib e no Vega — sem isso o
    # Vega empilharia em ordem alfabética do rótulo.
    g["ordem"] = g["oa_status"].map({s: i for i, s in enumerate(OA_STATUS_ORDEM)})
    g = g.sort_values(["ano", "ordem"])
    return g.reset_index(drop=True)


# --------------------------------------------------------------------------- #
# 4. APCs
# --------------------------------------------------------------------------- #
def indicadores_apc(df: pd.DataFrame) -> pd.DataFrame:
    """Custo de publicação por ano: total pago, média e cobertura do dado.

    Atenção: ``apc_paid`` só existe quando o OpenAlex consegue inferir o valor
    (em geral via DOAJ/OpenAPC). É um piso, não o gasto real.
    """
    if _vazio(df):
        return pd.DataFrame(columns=["ano", "obras", "obras_com_apc", "cobertura", "apc_total_usd", "apc_medio_usd"])
    d = df.copy()
    d["tem_apc"] = d["apc_pago_usd"].notna() & (d["apc_pago_usd"] > 0)
    g = d.groupby("ano", dropna=True).agg(
        obras=("id", "size"),
        obras_com_apc=("tem_apc", "sum"),
        apc_total_usd=("apc_pago_usd", "sum"),
        apc_medio_usd=("apc_pago_usd", "mean"),
    ).reset_index()
    g["cobertura"] = (100 * g["obras_com_apc"] / g["obras"]).round(2)
    g["apc_total_usd"] = g["apc_total_usd"].fillna(0).round(2)
    g["apc_medio_usd"] = g["apc_medio_usd"].fillna(0).round(2)
    g["ano"] = g["ano"].astype(int)
    return g[["ano", "obras", "obras_com_apc", "cobertura", "apc_total_usd", "apc_medio_usd"]]


def apc_por_editora(df: pd.DataFrame, top: int = 10) -> pd.DataFrame:
    """Para onde foi o dinheiro de APC."""
    if _vazio(df):
        return pd.DataFrame(columns=["editora", "obras_com_apc", "apc_total_usd", "apc_medio_usd"])
    d = df[df["apc_pago_usd"].notna() & (df["apc_pago_usd"] > 0)].copy()
    if d.empty:
        return pd.DataFrame(columns=["editora", "obras_com_apc", "apc_total_usd", "apc_medio_usd"])
    d["editora"] = d["fonte_editora"].fillna("(não informado)")
    g = d.groupby("editora").agg(
        obras_com_apc=("id", "size"),
        apc_total_usd=("apc_pago_usd", "sum"),
        apc_medio_usd=("apc_pago_usd", "mean"),
    ).reset_index()
    g[["apc_total_usd", "apc_medio_usd"]] = g[["apc_total_usd", "apc_medio_usd"]].round(2)
    return g.sort_values("apc_total_usd", ascending=False).head(top).reset_index(drop=True)


# --------------------------------------------------------------------------- #
# 5. Citação
# --------------------------------------------------------------------------- #
def citacoes_por_status(df: pd.DataFrame) -> pd.DataFrame:
    """Impacto de citação por via de acesso.

    A mediana é o número a ler: a média de citações é dominada por poucos
    outliers. E a diferença entre vias é associação, não causa — obras
    fechadas e abertas não são amostras comparáveis.
    """
    if _vazio(df):
        return pd.DataFrame(columns=["oa_status", "rotulo", "obras", "citacoes_media", "citacoes_mediana", "fwci_mediana"])
    g = df.groupby("oa_status").agg(
        obras=("id", "size"),
        citacoes_media=("citacoes", "mean"),
        citacoes_mediana=("citacoes", "median"),
        fwci_mediana=("fwci", "median"),
    ).reset_index()
    g["rotulo"] = g["oa_status"].map(OA_STATUS_ROTULO)
    g["_ordem"] = g["oa_status"].map({s: i for i, s in enumerate(OA_STATUS_ORDEM)})
    g = g.sort_values("_ordem").drop(columns="_ordem")
    for c in ("citacoes_media", "citacoes_mediana", "fwci_mediana"):
        g[c] = g[c].round(2)
    return g[["oa_status", "rotulo", "obras", "citacoes_media", "citacoes_mediana", "fwci_mediana"]].reset_index(drop=True)


# --------------------------------------------------------------------------- #
# 6. Fontes e áreas
# --------------------------------------------------------------------------- #
def top_fontes(df: pd.DataFrame, top: int = 15) -> pd.DataFrame:
    """Periódicos mais usados, com sua taxa de abertura e presença no DOAJ."""
    if _vazio(df):
        return pd.DataFrame(columns=["fonte", "editora", "obras", "taxa_oa", "no_doaj"])
    d = df[df["fonte"].notna()]
    if d.empty:
        return pd.DataFrame(columns=["fonte", "editora", "obras", "taxa_oa", "no_doaj"])
    g = d.groupby("fonte").agg(
        editora=("fonte_editora", "first"),
        obras=("id", "size"),
        taxa_oa=("is_oa", "mean"),
        no_doaj=("fonte_no_doaj", "first"),
    ).reset_index()
    g["taxa_oa"] = (100 * g["taxa_oa"]).round(1)
    return g.sort_values("obras", ascending=False).head(top).reset_index(drop=True)


def oa_por_area(df: pd.DataFrame, minimo: int = 10) -> pd.DataFrame:
    """Taxa de abertura por grande área, para áreas com massa suficiente."""
    if _vazio(df) or df["area"].isna().all():
        return pd.DataFrame(columns=["area", "obras", "taxa_oa"])
    g = df.dropna(subset=["area"]).groupby("area").agg(
        obras=("id", "size"), taxa_oa=("is_oa", "mean")
    ).reset_index()
    g["taxa_oa"] = (100 * g["taxa_oa"]).round(1)
    return g[g["obras"] >= minimo].sort_values("taxa_oa", ascending=False).reset_index(drop=True)


# --------------------------------------------------------------------------- #
# painel completo
# --------------------------------------------------------------------------- #
def painel_completo(df: pd.DataFrame) -> dict:
    """Roda todos os indicadores de uma vez. É o que alimenta o dashboard."""
    return {
        "gerais": indicadores_gerais(df),
        "distribuicao_oa": distribuicao_oa(df),
        "serie_anual_oa": serie_anual_oa(df),
        "serie_anual_status": serie_anual_por_status(df),
        "apc_anual": indicadores_apc(df),
        "apc_editora": apc_por_editora(df),
        "citacoes_status": citacoes_por_status(df),
        "top_fontes": top_fontes(df),
        "oa_area": oa_por_area(df),
    }


### Gráficos, specs Vega e dashboard

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.ticker import PercentFormatter


CORES_OA = {
    "diamond": "#2a78d6",  # azul
    "hybrid": "#e87ba4",   # rosa
    "gold": "#eda100",     # amarelo
    "green": "#1baf7a",    # verde-água
    "bronze": "#eb6834",   # laranja
    "closed": "#8f8e88",   # cinza neutro: ausência de abertura
}

TINTA_PRIMARIA = "#0b0b0b"
TINTA_SECUNDARIA = "#52514e"
GRADE = "#e4e3df"
AZUL = "#2a78d6"


def _estilo(ax, titulo: str, subtitulo: str = "") -> None:
    ax.set_title(titulo, loc="left", fontsize=13, color=TINTA_PRIMARIA,
                 pad=26 if subtitulo else 10)
    if subtitulo:
        ax.text(
            0, 1.012, subtitulo, transform=ax.transAxes,
            fontsize=9.5, color=TINTA_SECUNDARIA, va="bottom",
        )
    for lado in ("top", "right"):
        ax.spines[lado].set_visible(False)
    for lado in ("left", "bottom"):
        ax.spines[lado].set_color(GRADE)
    ax.tick_params(colors=TINTA_SECUNDARIA, labelsize=9, length=0)
    ax.grid(axis="y", color=GRADE, linewidth=0.8)
    ax.set_axisbelow(True)


def grafico_distribuicao_oa(dist: pd.DataFrame, titulo: str = "Distribuição por via de acesso"):
    """Barras horizontais: quanto cada via representa do total."""
    d = dist[dist["obras"] > 0].iloc[::-1]
    fig, ax = plt.subplots(figsize=(8, 0.55 * max(len(d), 3) + 1.6))
    ax.barh(d["rotulo"], d["percentual"],
            color=[CORES_OA[s] for s in d["oa_status"]], height=0.62)
    for y, (pct, n) in enumerate(zip(d["percentual"], d["obras"])):
        ax.text(pct + max(d["percentual"]) * 0.015, y, f"{pct:.1f}%  ({n:,})".replace(",", "."),
                va="center", fontsize=9, color=TINTA_SECUNDARIA)
    ax.set_xlim(0, max(d["percentual"]) * 1.28)
    ax.xaxis.set_major_formatter(PercentFormatter())
    ax.grid(axis="y", visible=False)
    ax.grid(axis="x", color=GRADE, linewidth=0.8)
    _estilo(ax, titulo, f"{int(d['obras'].sum()):,} obras".replace(",", "."))
    fig.tight_layout()
    return fig


def grafico_serie_oa(serie: pd.DataFrame, titulo: str = "Taxa de acesso aberto por ano"):
    """Linha única: evolução da taxa de abertura. Sem legenda — o título nomeia a série."""
    fig, ax = plt.subplots(figsize=(9, 4.4))
    ax.plot(serie["ano"], serie["taxa_oa"], color=AZUL, linewidth=2,
            marker="o", markersize=5, markerfacecolor=AZUL, markeredgecolor="white", markeredgewidth=1.5)
    # rótulo apenas nos extremos, não em todo ponto
    for i in (0, len(serie) - 1):
        if i < 0 or serie.empty:
            continue
        linha = serie.iloc[i]
        ax.annotate(f"{linha['taxa_oa']:.1f}%", (linha["ano"], linha["taxa_oa"]),
                    textcoords="offset points", xytext=(0, 10), ha="center",
                    fontsize=9.5, color=TINTA_PRIMARIA)
    ax.set_ylim(0, 100)
    ax.yaxis.set_major_formatter(PercentFormatter())
    ax.set_xticks(serie["ano"])
    _estilo(ax, titulo, "% das obras do período com alguma versão em acesso aberto")
    fig.tight_layout()
    return fig


def grafico_composicao_anual(longo: pd.DataFrame, titulo: str = "Composição das vias por ano"):
    """Barras empilhadas 100%: como a mistura de vias muda ao longo do tempo."""
    piv = longo.pivot_table(index="ano", columns="oa_status", values="percentual",
                            aggfunc="sum").fillna(0)
    piv = piv.reindex(columns=[s for s in OA_STATUS_ORDEM if s in piv.columns])
    fig, ax = plt.subplots(figsize=(9.5, 4.8))
    base = pd.Series(0.0, index=piv.index)
    for status in piv.columns:
        valores = piv[status]
        ax.bar(piv.index, valores, bottom=base, width=0.68,
               color=CORES_OA[status], label=OA_STATUS_ROTULO[status],
               edgecolor="white", linewidth=2)  # 2px de folga entre segmentos
        for x, (v, b) in zip(piv.index, zip(valores, base)):
            if v >= 7:  # rótulo direto só onde cabe
                ax.text(x, b + v / 2, f"{v:.0f}", ha="center", va="center",
                        fontsize=8.5, color="white" if status != "gold" else TINTA_PRIMARIA)
        base = base + valores
    ax.set_ylim(0, 100)
    ax.yaxis.set_major_formatter(PercentFormatter())
    ax.set_xticks(list(piv.index))
    ax.legend(frameon=False, fontsize=9, ncol=6, loc="upper center",
              bbox_to_anchor=(0.5, -0.11), labelcolor=TINTA_SECUNDARIA)
    _estilo(ax, titulo, "% das obras de cada ano")
    fig.tight_layout()
    return fig


def grafico_apc(apc: pd.DataFrame, titulo: str = "APCs pagos por ano (USD)"):
    """Uma medida por eixo: valor total. A cobertura do dado vai no rótulo."""
    fig, ax = plt.subplots(figsize=(9, 4.4))
    ax.bar(apc["ano"], apc["apc_total_usd"], width=0.62, color=AZUL)
    for _, r in apc.iterrows():
        ax.text(r["ano"], r["apc_total_usd"], f"US$ {r['apc_total_usd']:,.0f}".replace(",", "."),
                ha="center", va="bottom", fontsize=8.5, color=TINTA_SECUNDARIA)
    ax.set_xticks(list(apc["ano"]))
    cob = apc["cobertura"].mean() if len(apc) else 0
    _estilo(ax, titulo,
            f"Valor identificado pelo OpenAlex — é um piso. Cobertura média do dado: {cob:.0f}% das obras")
    fig.tight_layout()
    return fig


def grafico_citacoes(cit: pd.DataFrame, titulo: str = "Citações por via de acesso (mediana)"):
    """Mediana, não média: a média de citações é dominada por poucos outliers."""
    d = cit[cit["obras"] > 0]
    fig, ax = plt.subplots(figsize=(8, 4.2))
    ax.bar(d["rotulo"], d["citacoes_mediana"], width=0.6,
           color=[CORES_OA[s] for s in d["oa_status"]])
    for x, (v, n) in enumerate(zip(d["citacoes_mediana"], d["obras"])):
        ax.text(x, v, f"{v:g}\nn={n:,}".replace(",", "."), ha="center", va="bottom",
                fontsize=8.5, color=TINTA_SECUNDARIA)
    ax.margins(y=0.22)
    _estilo(ax, titulo, "Associação, não causa: obras abertas e fechadas não são amostras comparáveis")
    fig.tight_layout()
    return fig


def grafico_oa_por_area(area: pd.DataFrame, titulo: str = "Taxa de acesso aberto por área"):
    """Magnitude numa só dimensão: barras ordenadas, hue único."""
    d = area.sort_values("taxa_oa").tail(12)
    fig, ax = plt.subplots(figsize=(8.5, 0.42 * max(len(d), 4) + 1.6))
    ax.barh(d["area"], d["taxa_oa"], color=AZUL, height=0.62)
    for y, (v, n) in enumerate(zip(d["taxa_oa"], d["obras"])):
        ax.text(v + 1.2, y, f"{v:.0f}%  (n={n:,})".replace(",", "."), va="center",
                fontsize=8.5, color=TINTA_SECUNDARIA)
    ax.set_xlim(0, 112)
    ax.xaxis.set_major_formatter(PercentFormatter())
    ax.grid(axis="y", visible=False)
    ax.grid(axis="x", color=GRADE, linewidth=0.8)
    _estilo(ax, titulo)
    fig.tight_layout()
    return fig


import json
from typing import Any

import pandas as pd


SUPERFICIE = "#fcfcfb"
TINTA_PRIMARIA = "#0b0b0b"
TINTA_SECUNDARIA = "#52514e"
GRADE = "#e4e3df"
AZUL = "#2a78d6"

SCHEMA = "https://vega.github.io/schema/vega/v5.json"

_ROTULOS = [OA_STATUS_ROTULO[s] for s in OA_STATUS_ORDEM]
_CORES = [CORES_OA[s] for s in OA_STATUS_ORDEM]


def _base(width: int, height: int) -> dict[str, Any]:
    return {
        "$schema": SCHEMA,
        "width": width,
        "height": height,
        "padding": 8,
        "autosize": {"type": "fit", "contains": "padding"},
        "background": SUPERFICIE,
        "config": {
            "axis": {
                "labelColor": TINTA_SECUNDARIA,
                "labelFontSize": 11,
                "titleColor": TINTA_SECUNDARIA,
                "titleFontSize": 11,
                "titleFontWeight": "normal",
                "domainColor": GRADE,
                "tickColor": GRADE,
                "gridColor": GRADE,
                "gridWidth": 1,
            },
            "legend": {
                "labelColor": TINTA_SECUNDARIA,
                "labelFontSize": 11,
                "titleColor": TINTA_SECUNDARIA,
                "symbolType": "square",
                "symbolSize": 90,
            },
        },
    }


def _escala_cor_oa() -> dict[str, Any]:
    return {
        "name": "cor",
        "type": "ordinal",
        "domain": _ROTULOS,
        "range": _CORES,
    }


# --------------------------------------------------------------------------- #
# 1. distribuição por via de acesso — barras horizontais
# --------------------------------------------------------------------------- #
def spec_distribuicao(dist: pd.DataFrame, width: int = 520) -> dict:
    dados = dist[dist["obras"] > 0].to_dict("records")
    altura = max(len(dados), 3) * 34
    spec = _base(width, altura)
    spec["data"] = [{"name": "tabela", "values": dados}]
    spec["scales"] = [
        {
            "name": "y",
            "type": "band",
            "domain": [r["rotulo"] for r in dados],
            "range": "height",
            "padding": 0.32,
        },
        {
            "name": "x",
            "type": "linear",
            "domain": {"data": "tabela", "field": "percentual"},
            "range": "width",
            "nice": True,
            "zero": True,
        },
        _escala_cor_oa(),
    ]
    spec["axes"] = [
        {"orient": "left", "scale": "y", "domain": False, "ticks": False, "labelPadding": 8},
        {"orient": "bottom", "scale": "x", "grid": True, "format": "d", "title": "% das obras"},
    ]
    spec["marks"] = [
        {
            "type": "rect",
            "from": {"data": "tabela"},
            "encode": {
                "enter": {
                    "y": {"scale": "y", "field": "rotulo"},
                    "height": {"scale": "y", "band": 1},
                    "x": {"scale": "x", "value": 0},
                    "x2": {"scale": "x", "field": "percentual"},
                    "fill": {"scale": "cor", "field": "rotulo"},
                    "cornerRadiusTopRight": {"value": 4},
                    "cornerRadiusBottomRight": {"value": 4},
                    "tooltip": {
                        "signal": "datum.rotulo + ': ' + format(datum.percentual, '.1f') "
                        "+ '% (' + format(datum.obras, ',d') + ' obras)'"
                    },
                },
                "update": {"fillOpacity": {"value": 1}},
                "hover": {"fillOpacity": {"value": 0.78}},
            },
        },
        {
            "type": "text",
            "from": {"data": "tabela"},
            "encode": {
                "enter": {
                    "y": {"scale": "y", "field": "rotulo", "band": 0.5},
                    "x": {"scale": "x", "field": "percentual", "offset": 8},
                    "text": {"signal": "format(datum.percentual, '.1f') + '%'"},
                    "baseline": {"value": "middle"},
                    "fontSize": {"value": 11},
                    "fill": {"value": TINTA_SECUNDARIA},
                }
            },
        },
    ]
    return spec


# --------------------------------------------------------------------------- #
# 2. taxa de OA por ano — linha
# --------------------------------------------------------------------------- #
def spec_serie_oa(serie: pd.DataFrame, width: int = 520, height: int = 260) -> dict:
    dados = serie.to_dict("records")
    spec = _base(width, height)
    spec["data"] = [{"name": "tabela", "values": dados}]
    spec["scales"] = [
        {
            "name": "x",
            "type": "point",
            "domain": [r["ano"] for r in dados],
            "range": "width",
            "padding": 0.5,
        },
        {"name": "y", "type": "linear", "domain": [0, 100], "range": "height", "zero": True},
    ]
    spec["axes"] = [
        {"orient": "bottom", "scale": "x", "format": "d", "labelAngle": 0},
        {"orient": "left", "scale": "y", "grid": True, "title": "% em acesso aberto"},
    ]
    spec["marks"] = [
        {
            "type": "line",
            "from": {"data": "tabela"},
            "encode": {
                "enter": {
                    "x": {"scale": "x", "field": "ano"},
                    "y": {"scale": "y", "field": "taxa_oa"},
                    "stroke": {"value": AZUL},
                    "strokeWidth": {"value": 2},
                }
            },
        },
        {
            "type": "symbol",
            "from": {"data": "tabela"},
            "encode": {
                "enter": {
                    "x": {"scale": "x", "field": "ano"},
                    "y": {"scale": "y", "field": "taxa_oa"},
                    "fill": {"value": AZUL},
                    "stroke": {"value": SUPERFICIE},
                    "strokeWidth": {"value": 1.5},
                    "size": {"value": 70},
                    "tooltip": {
                        "signal": "datum.ano + ': ' + format(datum.taxa_oa, '.1f') + '% de ' "
                        "+ format(datum.obras, ',d') + ' obras'"
                    },
                },
                "hover": {"size": {"value": 140}},
            },
        },
    ]
    return spec


# --------------------------------------------------------------------------- #
# 3. composição das vias por ano — barras empilhadas 100%
# --------------------------------------------------------------------------- #
def spec_composicao(longo: pd.DataFrame, width: int = 520, height: int = 280) -> dict:
    dados = longo.to_dict("records")
    spec = _base(width, height)
    spec["data"] = [
        {
            "name": "tabela",
            "values": dados,
            "transform": [
                {
                    "type": "stack",
                    "groupby": ["ano"],
                    "field": "percentual",
                    # ordem semântica das vias, não alfabética do rótulo
                    "sort": {"field": "ordem", "order": "ascending"},
                }
            ],
        }
    ]
    spec["scales"] = [
        {
            "name": "x",
            "type": "band",
            "domain": sorted({r["ano"] for r in dados}),
            "range": "width",
            "padding": 0.28,
        },
        {"name": "y", "type": "linear", "domain": [0, 100], "range": "height", "zero": True},
        _escala_cor_oa(),
    ]
    spec["axes"] = [
        {"orient": "bottom", "scale": "x", "format": "d", "labelAngle": 0},
        {"orient": "left", "scale": "y", "grid": True, "title": "% das obras do ano"},
    ]
    spec["legends"] = [{"fill": "cor", "orient": "bottom", "direction": "horizontal", "columns": 6}]
    spec["marks"] = [
        {
            "type": "rect",
            "from": {"data": "tabela"},
            "encode": {
                "enter": {
                    "x": {"scale": "x", "field": "ano"},
                    "width": {"scale": "x", "band": 1},
                    "y": {"scale": "y", "field": "y0"},
                    "y2": {"scale": "y", "field": "y1"},
                    "fill": {"scale": "cor", "field": "rotulo"},
                    "stroke": {"value": SUPERFICIE},
                    "strokeWidth": {"value": 2},
                    "tooltip": {
                        "signal": "datum.ano + ' · ' + datum.rotulo + ': ' "
                        "+ format(datum.percentual, '.1f') + '% (' + format(datum.obras, ',d') + ')'"
                    },
                },
                "update": {"fillOpacity": {"value": 1}},
                "hover": {"fillOpacity": {"value": 0.78}},
            },
        }
    ]
    return spec


# --------------------------------------------------------------------------- #
# 4. APC por ano — barras
# --------------------------------------------------------------------------- #
def spec_apc(apc: pd.DataFrame, width: int = 520, height: int = 260) -> dict:
    dados = apc.to_dict("records")
    spec = _base(width, height)
    spec["data"] = [{"name": "tabela", "values": dados}]
    spec["scales"] = [
        {
            "name": "x",
            "type": "band",
            "domain": [r["ano"] for r in dados],
            "range": "width",
            "padding": 0.3,
        },
        {
            "name": "y",
            "type": "linear",
            "domain": {"data": "tabela", "field": "apc_total_usd"},
            "range": "height",
            "nice": True,
            "zero": True,
        },
    ]
    spec["axes"] = [
        {"orient": "bottom", "scale": "x", "format": "d", "labelAngle": 0},
        {"orient": "left", "scale": "y", "grid": True, "title": "APC pago (USD)", "format": "~s"},
    ]
    spec["marks"] = [
        {
            "type": "rect",
            "from": {"data": "tabela"},
            "encode": {
                "enter": {
                    "x": {"scale": "x", "field": "ano"},
                    "width": {"scale": "x", "band": 1},
                    "y": {"scale": "y", "field": "apc_total_usd"},
                    "y2": {"scale": "y", "value": 0},
                    "fill": {"value": AZUL},
                    "cornerRadiusTopLeft": {"value": 4},
                    "cornerRadiusTopRight": {"value": 4},
                    "tooltip": {
                        "signal": "datum.ano + ': US$ ' + format(datum.apc_total_usd, ',.0f') "
                        "+ ' em ' + format(datum.obras_com_apc, ',d') + ' obras (cobertura ' "
                        "+ format(datum.cobertura, '.0f') + '%)'"
                    },
                },
                "update": {"fillOpacity": {"value": 1}},
                "hover": {"fillOpacity": {"value": 0.78}},
            },
        }
    ]
    return spec


# --------------------------------------------------------------------------- #
# 5. citações por via — barras
# --------------------------------------------------------------------------- #
def spec_citacoes(cit: pd.DataFrame, width: int = 520, height: int = 260) -> dict:
    dados = cit[cit["obras"] > 0].to_dict("records")
    spec = _base(width, height)
    spec["data"] = [{"name": "tabela", "values": dados}]
    spec["scales"] = [
        {
            "name": "x",
            "type": "band",
            "domain": [r["rotulo"] for r in dados],
            "range": "width",
            "padding": 0.3,
        },
        {
            "name": "y",
            "type": "linear",
            "domain": {"data": "tabela", "field": "citacoes_mediana"},
            "range": "height",
            "nice": True,
            "zero": True,
        },
        _escala_cor_oa(),
    ]
    spec["axes"] = [
        {"orient": "bottom", "scale": "x"},
        {"orient": "left", "scale": "y", "grid": True, "title": "citações (mediana)"},
    ]
    spec["marks"] = [
        {
            "type": "rect",
            "from": {"data": "tabela"},
            "encode": {
                "enter": {
                    "x": {"scale": "x", "field": "rotulo"},
                    "width": {"scale": "x", "band": 1},
                    "y": {"scale": "y", "field": "citacoes_mediana"},
                    "y2": {"scale": "y", "value": 0},
                    "fill": {"scale": "cor", "field": "rotulo"},
                    "cornerRadiusTopLeft": {"value": 4},
                    "cornerRadiusTopRight": {"value": 4},
                    "tooltip": {
                        "signal": "datum.rotulo + ': mediana ' + datum.citacoes_mediana "
                        "+ ' citações em ' + format(datum.obras, ',d') + ' obras'"
                    },
                },
                "update": {"fillOpacity": {"value": 1}},
                "hover": {"fillOpacity": {"value": 0.78}},
            },
        }
    ]
    return spec


# --------------------------------------------------------------------------- #
def todas_as_specs(painel: dict) -> dict[str, dict]:
    """Recebe a saída de ``indicators.painel_completo`` e devolve as specs."""
    return {
        "distribuicao": spec_distribuicao(painel["distribuicao_oa"]),
        "serie_oa": spec_serie_oa(painel["serie_anual_oa"]),
        "composicao": spec_composicao(painel["serie_anual_status"]),
        "apc": spec_apc(painel["apc_anual"]),
        "citacoes": spec_citacoes(painel["citacoes_status"]),
    }


def salvar_specs(specs: dict[str, dict], caminho: str) -> str:
    with open(caminho, "w", encoding="utf-8") as fh:
        json.dump(specs, fh, ensure_ascii=False, indent=2)
    return caminho


import html
import json
from datetime import datetime

import pandas as pd


_CSS = """
:root{color-scheme:light;--surface-0:#f5f4f1;--surface-1:#fcfcfb;--line:#e4e3df;
--ink-1:#0b0b0b;--ink-2:#52514e;--ink-3:#7a7973;--accent:#2a78d6}
*{box-sizing:border-box}
body{margin:0;background:var(--surface-0);color:var(--ink-1);
font:14px/1.55 ui-sans-serif,system-ui,-apple-system,"Segoe UI",Roboto,Helvetica,Arial,sans-serif}
.wrap{max-width:1180px;margin:0 auto;padding:32px 20px 64px}
header h1{font-size:24px;margin:0 0 6px;letter-spacing:-.01em}
header p{margin:0;color:var(--ink-2);font-size:13.5px}
.meta{margin-top:10px;font-size:12.5px;color:var(--ink-3)}
.kpis{display:grid;grid-template-columns:repeat(auto-fit,minmax(170px,1fr));gap:12px;margin:26px 0}
.kpi{background:var(--surface-1);border:1px solid var(--line);border-radius:10px;padding:14px 16px}
.kpi .n{font-size:25px;font-weight:600;letter-spacing:-.02em;line-height:1.15}
.kpi .l{font-size:12px;color:var(--ink-2);margin-top:4px}
.kpi .s{font-size:11.5px;color:var(--ink-3);margin-top:2px}
.grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(430px,1fr));gap:16px}
.card{background:var(--surface-1);border:1px solid var(--line);border-radius:10px;padding:18px 18px 14px;min-width:0}
.card h2{font-size:15px;margin:0 0 2px;font-weight:600}
.card p.sub{margin:0 0 14px;font-size:12.5px;color:var(--ink-2)}
.chart{width:100%;overflow-x:auto}
details{margin-top:10px;border-top:1px solid var(--line);padding-top:8px}
summary{font-size:12.5px;color:var(--ink-2);cursor:pointer}
table{border-collapse:collapse;width:100%;margin-top:10px;font-size:12.5px}
th,td{text-align:right;padding:5px 8px;border-bottom:1px solid var(--line);white-space:nowrap}
th:first-child,td:first-child{text-align:left}
th{color:var(--ink-2);font-weight:600}
footer{margin-top:32px;font-size:12px;color:var(--ink-3);line-height:1.7}
footer code{background:var(--surface-1);border:1px solid var(--line);border-radius:4px;padding:1px 5px}
"""


def _fmt(n, casas: int = 1) -> str:
    """Formata no padrão pt-BR: 1.234,5"""
    if isinstance(n, float) and casas:
        return f"{n:,.{casas}f}".replace(",", "@").replace(".", ",").replace("@", ".")
    return f"{round(n or 0):,}".replace(",", ".")


def _tabela(df: pd.DataFrame, colunas: dict[str, str]) -> str:
    if df is None or df.empty:
        return "<p style='font-size:12.5px;color:#7a7973'>Sem dados.</p>"
    d = df[[c for c in colunas if c in df.columns]]
    cab = "".join(f"<th>{html.escape(colunas[c])}</th>" for c in d.columns)
    linhas = "".join(
        "<tr>" + "".join(f"<td>{html.escape(str(v))}</td>" for v in row) + "</tr>"
        for row in d.itertuples(index=False)
    )
    return f"<table><thead><tr>{cab}</tr></thead><tbody>{linhas}</tbody></table>"


def _kpi(numero: str, rotulo: str, sub: str = "") -> str:
    extra = f"<div class='s'>{html.escape(sub)}</div>" if sub else ""
    return (
        f"<div class='kpi'><div class='n'>{html.escape(numero)}</div>"
        f"<div class='l'>{html.escape(rotulo)}</div>{extra}</div>"
    )


def _card(titulo: str, sub: str, chart_id: str, tabela_html: str) -> str:
    return f"""<section class="card">
  <h2>{html.escape(titulo)}</h2>
  <p class="sub">{html.escape(sub)}</p>
  <div class="chart" id="{chart_id}"></div>
  <details><summary>Ver os dados em tabela</summary>{tabela_html}</details>
</section>"""


def gerar_dashboard(
    painel: dict,
    titulo: str,
    subtitulo: str = "",
    caminho: str = "dashboard_oa.html",
) -> str:
    """Escreve o dashboard e devolve o caminho do arquivo."""
    g = painel.get("gerais") or {}
    specs = todas_as_specs(painel)

    kpis = "".join([
        _kpi(f"{g.get('taxa_oa', 0)}%", "Acesso aberto",
             f"{_fmt(g.get('obras_oa', 0))} de {_fmt(g.get('obras', 0))} obras"),
        _kpi(f"{g.get('taxa_diamante', 0)}%", "Via diamante", "sem cobrança de APC ao autor"),
        _kpi(f"{g.get('taxa_em_repositorio', 0)}%", "Em repositório", "texto completo depositado"),
        _kpi(f"{g.get('taxa_no_doaj', 0)}%", "Em periódico do DOAJ"),
        _kpi(f"US$ {_fmt(g.get('apc_total_usd', 0), casas=0)}", "APC identificado",
             f"em {_fmt(g.get('obras_com_apc_pago', 0))} obras — é um piso"),
        _kpi(_fmt(g.get("citacoes_por_obra", 0)), "Citações por obra",
             f"{_fmt(g.get('citacoes_totais', 0))} no total"),
    ])

    cards = "".join([
        _card("Distribuição por via de acesso",
              "Cada via tem cor fixa em todo o painel.",
              "c-distribuicao",
              _tabela(painel["distribuicao_oa"],
                      {"rotulo": "Via", "obras": "Obras", "percentual": "%"})),
        _card("Taxa de acesso aberto por ano",
              "% das obras do ano com alguma versão aberta.",
              "c-serie",
              _tabela(painel["serie_anual_oa"],
                      {"ano": "Ano", "obras": "Obras", "obras_oa": "Abertas", "taxa_oa": "% aberto"})),
        _card("Composição das vias por ano",
              "Como a mistura de vias muda ao longo do tempo.",
              "c-composicao",
              _tabela(painel["serie_anual_status"],
                      {"ano": "Ano", "rotulo": "Via", "obras": "Obras", "percentual": "%"})),
        _card("APCs pagos por ano",
              "Valor que o OpenAlex consegue identificar — piso, não gasto real.",
              "c-apc",
              _tabela(painel["apc_anual"],
                      {"ano": "Ano", "obras_com_apc": "Obras com APC", "cobertura": "Cobertura %",
                       "apc_total_usd": "Total USD", "apc_medio_usd": "Médio USD"})),
        _card("Citações por via de acesso",
              "Mediana. Associação, não causa: as amostras não são comparáveis.",
              "c-citacoes",
              _tabela(painel["citacoes_status"],
                      {"rotulo": "Via", "obras": "Obras", "citacoes_mediana": "Mediana",
                       "citacoes_media": "Média", "fwci_mediana": "FWCI mediano"})),
        _card("Periódicos mais usados",
              "Onde a produção foi publicada, com a taxa de abertura de cada título.",
              "c-fontes",
              _tabela(painel["top_fontes"],
                      {"fonte": "Periódico", "editora": "Editora", "obras": "Obras",
                       "taxa_oa": "% aberto", "no_doaj": "DOAJ"})),
    ])

    specs_json = json.dumps(specs, ensure_ascii=False)
    agora = datetime.now().strftime("%d/%m/%Y às %H:%M")

    return _escrever(caminho, f"""<!doctype html>
<html lang="pt-BR">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>{html.escape(titulo)}</title>
<style>{_CSS}</style>
</head>
<body>
<div class="wrap">
  <header>
    <h1>{html.escape(titulo)}</h1>
    <p>{html.escape(subtitulo)}</p>
    <div class="meta">Período {html.escape(str(g.get('periodo', '—')))} ·
      {_fmt(g.get('obras', 0))} obras · dados do OpenAlex · gerado em {agora}</div>
  </header>
  <div class="kpis">{kpis}</div>
  <div class="grid">{cards}</div>
  <footer>
    Fonte dos dados: <strong>OpenAlex</strong> (CC0), via API pública.
    Gerado com <code>OA_monitor</code> — Workshop ConfOA 2026.<br>
    A cobertura de APC e de vínculos institucionais no OpenAlex é parcial; leia os
    valores de APC como um piso e confira os totais contra a fonte antes de publicá-los.
  </footer>
</div>
<script src="https://cdnjs.cloudflare.com/ajax/libs/vega/5.30.0/vega.min.js"></script>
<script>
  const SPECS = {specs_json};
  const ALVOS = {{
    "c-distribuicao": "distribuicao", "c-serie": "serie_oa",
    "c-composicao": "composicao", "c-apc": "apc", "c-citacoes": "citacoes"
  }};
  function desenhar() {{
    for (const [elId, specId] of Object.entries(ALVOS)) {{
      const el = document.getElementById(elId);
      const spec = SPECS[specId];
      if (!el || !spec) continue;
      el.innerHTML = "";
      try {{
        const largura = Math.max(320, el.clientWidth || 480);
        const view = new vega.View(vega.parse({{...spec, width: largura - 10}}), {{
          renderer: "canvas", container: el, hover: true
        }});
        view.runAsync();
      }} catch (e) {{ console.error("Falha ao desenhar " + specId, e); }}
    }}
  }}
  const cartaoFontes = document.getElementById("c-fontes");
  if (cartaoFontes) {{
    const d = cartaoFontes.closest(".card").querySelector("details");
    if (d) d.open = true;   // este cartão é só tabela
  }}
  window.addEventListener("load", desenhar);
  let t; window.addEventListener("resize", () => {{ clearTimeout(t); t = setTimeout(desenhar, 200); }});
</script>
</body>
</html>
""")


def _escrever(caminho: str, conteudo: str) -> str:
    with open(caminho, "w", encoding="utf-8") as fh:
        fh.write(conteudo)
    return caminho


---
## 0.3 Sua credencial individual do OpenAlex

**Cada participante usa a sua própria credencial.** São duas coisas diferentes:

| | O que é | Como obter | Obrigatório? |
|---|---|---|---|
| **`mailto`** | seu e-mail, enviado em cada requisição | você já tem | **sim** |
| **`api_key`** | chave pessoal, com cota diária só sua | conta gratuita em [openalex.org](https://openalex.org) → *Settings* → *API key* | recomendada |

Informar o e-mail coloca você no *polite pool* do OpenAlex — respostas mais estáveis. A chave é gratuita para a maior parte dos usos e dá a você uma cota diária individual.

**Por que isso importa numa sala de aula:** sem chave, todos os participantes saem pelo mesmo IP da rede local e **dividem uma única cota** — depois de algumas dezenas de consultas a API começa a recusar as requisições de todo mundo. Com a chave individual, cada um tem a sua.

> 🔒 **Nunca digite a chave direto numa célula e nunca a suba para o GitHub.** A célula abaixo lê a chave dos *Secrets* do Colab (ícone 🔑 na barra lateral, nome `OPENALEX_API_KEY`) ou pergunta em campo oculto.

In [ ]:
# @title Preencha o seu e-mail e rode esta célula
EMAIL = ""  # @param {type:"string", placeholder:"seu.email@instituicao.br"}
USAR_API_KEY = True  # @param {type:"boolean"}

from getpass import getpass

_chave = None
if USAR_API_KEY:
    try:  # 1ª opção: Secrets do Colab (🔑 na barra lateral)
        from google.colab import userdata
        _chave = userdata.get('OPENALEX_API_KEY')
        print('Chave lida dos Secrets do Colab.')
    except Exception:
        _chave = getpass('Cole sua OpenAlex API key (Enter para seguir sem ela): ') or None

credenciais = Credentials(mailto=EMAIL, api_key=_chave)
cliente = OpenAlexClient(credenciais)
print('Credencial configurada →', credenciais.describe())


---
# Parte 2 — Atividades práticas

## 2.1 Pesquisa e extração de dados

### Escolha o seu recorte

| Nível | O que preencher em `IDENTIFICADOR` | Onde achar | Exemplo |
|---|---|---|---|
| `pais` | código ISO de 2 letras | — | `BR`, `PT`, `AR` |
| `instituicao` | **o ROR da instituição** | [ror.org](https://ror.org) — busque pelo nome | `041akq887` (UFSC) |
| `pesquisador` | o ORCID | [orcid.org](https://orcid.org) | `0000-0002-8338-1931` |
| `fonte` | o ISSN do periódico | [portal.issn.org](https://portal.issn.org) | `1518-2924` |

O ROR pode ser colado inteiro (`https://ror.org/041akq887`) ou só o código (`041akq887`) — a função normaliza os dois.

**Alguns RORs para testar:** UFSC `041akq887` · IBICT `04sk9pv44` · CEFET-MG `0384j8v12` · USP `036rp1748` · Univ. do Minho `037wpkx04`

In [ ]:
# @title Defina o recorte da análise
NIVEL = "instituicao"  # @param ["pais", "instituicao", "pesquisador", "fonte"]
IDENTIFICADOR = "041akq887"  # @param {type:"string", placeholder:"digite o ROR da instituição"}
ANO_INICIO = 2020  # @param {type:"integer"}
ANO_FIM = 2024  # @param {type:"integer"}
SOMENTE_ARTIGOS = True  # @param {type:"boolean"}

tipos = ['article', 'review'] if SOMENTE_ARTIGOS else None
filtro = montar_filtro(NIVEL, IDENTIFICADOR, ANO_INICIO, ANO_FIM, tipos=tipos)
print('filter =', filtro)
print()
print('URL equivalente no navegador:')
print(f'https://api.openalex.org/works?filter={filtro}')


### Dimensione antes de baixar

Duas requisições respondem *"quantos registros existem?"* e *"como se distribuem por via de acesso?"* **sem baixar nada**. O OpenAlex agrega no servidor (`group_by`).

Use isto para decidir o `MAX_REGISTROS` da célula seguinte: a extração baixa 200 registros por requisição, então 20 mil obras ≈ 100 requisições ≈ 1–2 minutos.

In [ ]:
resumo = resumo_rapido(cliente, filtro)
print(f"Total de obras no recorte: {resumo['total']:,}".replace(',', '.'))
print()
for status, n in sorted(resumo['por_oa_status'].items(), key=lambda x: -x[1]):
    pct = 100 * n / max(resumo['total'], 1)
    rotulo = OA_STATUS_ROTULO.get(status, status)
    print(f'  {rotulo:<10} {n:>8,}  {pct:5.1f}%'.replace(',', '.'))


### Extraia

Se o total acima for grande, comece com `MAX_REGISTROS = 2000` para a aula andar; depois rode o recorte inteiro em casa.

In [ ]:
# @title Baixar os dados
MAX_REGISTROS = 3000  # @param {type:"integer"}

def _progresso(baixados, total):
    print(f'\r  {baixados:,} / {total:,} obras'.replace(',', '.'), end='')

t0 = time.time()
df = extrair_works(cliente, filtro, max_registros=MAX_REGISTROS, on_progress=_progresso)
print(f'\n\nPronto: {len(df):,} obras em {time.time() - t0:.0f}s'.replace(',', '.'))
if cliente.last_rate_limit:
    print('Cota:', cliente.last_rate_limit)
df.head(3)


### Confira o que veio

Antes de calcular qualquer indicador, olhe os dados. Quantos anos? Quantos sem periódico identificado? A cobertura de APC é grande o bastante para o número significar alguma coisa?

In [ ]:
print(f'Obras: {len(df):,}'.replace(',', '.'))
print(f"Período: {df['ano'].min()}–{df['ano'].max()}")
print(f"Sem periódico identificado: {df['fonte'].isna().sum():,}".replace(',', '.'))
print(f"Com valor de APC pago: {df['apc_pago_usd'].notna().sum():,} "
      f"({100 * df['apc_pago_usd'].notna().mean():.1f}% das obras)".replace(',', '.'))
print()
display(df['tipo'].value_counts().head())
display(df.dtypes.to_frame('tipo do campo').T)


In [ ]:
# Guarde o bruto — extrair de novo custa tempo e cota
os.makedirs('dados', exist_ok=True)
arquivo_bruto = f'dados/works_{NIVEL}_{ANO_INICIO}_{ANO_FIM}.csv'
df.to_csv(arquivo_bruto, index=False)
print('Salvo em', arquivo_bruto)


---
## 2.2 Geração de indicadores

`painel_completo` roda todos os indicadores de uma vez e devolve um dicionário de DataFrames. Cada um também pode ser chamado isoladamente.

In [ ]:
painel = painel_completo(df)

for chave, valor in painel['gerais'].items():
    print(f'{chave:<24} {valor}')


### Como ler estes números

- **taxa_oa** — % de obras com alguma versão em acesso aberto (qualquer via).
- **taxa_diamante** — publicadas em periódicos abertos que **não cobram APC do autor**. É o indicador mais próximo de "ciência aberta sem custo para quem publica".
- **taxa_em_repositorio** — % com texto completo depositado em repositório (via verde). Para um gestor de repositório, é *o* indicador.
- **apc_total_usd** — ⚠️ **é um piso, não o gasto real.** O OpenAlex só conhece o valor quando consegue inferi-lo (via DOAJ/OpenAPC); acordos institucionais, descontos e isenções não aparecem. Sempre reporte junto a **cobertura** do dado.
- **citacoes_por_obra** — a média é puxada por poucos casos extremos; nos gráficos usamos a **mediana**.

In [ ]:
# Distribuição por via de acesso
painel['distribuicao_oa']


In [ ]:
# Evolução anual
painel['serie_anual_oa']


In [ ]:
# APCs: repare na coluna `cobertura` antes de olhar o total
painel['apc_anual']


In [ ]:
# Para onde foi o dinheiro
painel['apc_editora']


In [ ]:
# Citação por via — leia a mediana, e leia como associação, não como causa
painel['citacoes_status']


In [ ]:
# Onde a produção foi publicada
painel['top_fontes']


---
## 2.3 Visualização de dados

Cada via de acesso tem **uma cor fixa** em todo o material — a cor identifica a via, nunca a posição no gráfico. As cinco vias abertas usam matizes distintos e "Fechado" é cinza: é a ausência de abertura, não uma sexta via.

A ordem (diamante → híbrido → dourado → verde → bronze) foi escolhida para que cores vizinhas nas barras empilhadas continuem distinguíveis por leitores com daltonismo. Por isso não troque a ordem sem verificar o contraste.

In [ ]:
fig = grafico_distribuicao_oa(painel['distribuicao_oa'])
plt.show()


In [ ]:
fig = grafico_serie_oa(painel['serie_anual_oa'])
plt.show()


In [ ]:
fig = grafico_composicao_anual(painel['serie_anual_status'])
plt.show()


In [ ]:
fig = grafico_apc(painel['apc_anual'])
plt.show()


In [ ]:
fig = grafico_citacoes(painel['citacoes_status'])
plt.show()


In [ ]:
if not painel['oa_area'].empty:
    fig = grafico_oa_por_area(painel['oa_area'])
    plt.show()
else:
    print('Poucas obras por área neste recorte.')


### Salvar as figuras

In [ ]:
os.makedirs('figuras', exist_ok=True)
figuras = {
    'distribuicao_oa': grafico_distribuicao_oa(painel['distribuicao_oa']),
    'serie_oa': grafico_serie_oa(painel['serie_anual_oa']),
    'composicao_anual': grafico_composicao_anual(painel['serie_anual_status']),
    'apc_anual': grafico_apc(painel['apc_anual']),
    'citacoes_status': grafico_citacoes(painel['citacoes_status']),
}
for nome, figura in figuras.items():
    figura.savefig(f'figuras/{nome}.png', dpi=200, bbox_inches='tight', facecolor='#fcfcfb')
plt.close('all')
print('Figuras salvas em figuras/')


---
# Parte 3 — Dashboard

Os mesmos indicadores, agora num painel interativo montado com [**Vega**](https://vega.github.io/vega/): passe o mouse sobre qualquer marca para ver os números, e cada gráfico traz a tabela equivalente logo abaixo.

O arquivo gerado é **autocontido** — os dados vão embutidos nele. Abre com um duplo clique, funciona offline (menos a biblioteca Vega, que vem do CDN) e pode ser enviado por e-mail ou publicado num site institucional.

As mesmas *specs* Vega são consumidas pelo componente React [`dashboard/VegaChart.tsx`](https://github.com/fabiocantoadv/OA_monitor/blob/main/dashboard/VegaChart.tsx) do repositório, se você quiser embutir os gráficos numa aplicação sua.

In [ ]:
# @title Gerar o dashboard
TITULO_DASHBOARD = "Indicadores de Ciência Aberta"  # @param {type:"string"}
SUBTITULO_DASHBOARD = ""  # @param {type:"string", placeholder:"ex.: UFSC, 2020–2024"}

subtitulo = SUBTITULO_DASHBOARD or f'{NIVEL}: {IDENTIFICADOR} · {ANO_INICIO}–{ANO_FIM}'
arquivo_dash = 'dashboard_oa.html'
gerar_dashboard(painel, TITULO_DASHBOARD, subtitulo, arquivo_dash)

tamanho = os.path.getsize(arquivo_dash) / 1024
print(f'Dashboard gerado: {arquivo_dash} ({tamanho:.0f} KB)')


### Ver aqui no notebook

In [ ]:
from IPython.display import HTML, display

display(HTML(
    f'<iframe srcdoc="{html.escape(open(arquivo_dash, encoding="utf-8").read())}" '
    'style="width:100%;height:820px;border:1px solid #e4e3df;border-radius:8px"></iframe>'
))


### Exportar tudo

Além do HTML, vale levar uma planilha com os indicadores (uma aba por tabela) e as *specs* Vega em JSON, para reaproveitar os mesmos gráficos noutro lugar.

In [ ]:
import shutil

# specs Vega separadas, para reaproveitar em outra aplicação
salvar_specs(todas_as_specs(painel), 'specs_vega.json')

# planilha com todos os indicadores, uma aba por tabela
with pd.ExcelWriter('indicadores_oa.xlsx') as writer:
    pd.DataFrame([painel['gerais']]).T.rename(columns={0: 'valor'}).to_excel(
        writer, sheet_name='geral')
    for nome, tabela in painel.items():
        if isinstance(tabela, pd.DataFrame) and not tabela.empty:
            tabela.to_excel(writer, sheet_name=nome[:31], index=False)

shutil.make_archive('figuras_oa', 'zip', 'figuras')

arquivos = [arquivo_dash, 'indicadores_oa.xlsx', 'specs_vega.json',
            'figuras_oa.zip', arquivo_bruto]
for arquivo in arquivos:
    print(f'{os.path.getsize(arquivo) / 1024:8.0f} KB  {arquivo}')


### Baixar para o seu computador

In [ ]:
try:
    from google.colab import files
    for arquivo in arquivos:
        files.download(arquivo)
except ImportError:
    print('Fora do Colab: os arquivos já estão na pasta de trabalho.')


---
# Parte 4 — Adapte ao seu contexto

### Exercícios

1. **Compare duas instituições.** Rode a extração duas vezes com RORs diferentes e coloque as duas séries no mesmo gráfico. *(Dica: dois gráficos lado a lado costumam ler melhor que duas linhas sobrepostas quando os volumes são muito diferentes.)*
2. **Recorte por área.** Filtre `df` por `area` e refaça o painel — a taxa de abertura de Saúde e a de Humanidades quase nunca se parecem.
3. **Sua própria produção.** Troque `NIVEL` para `pesquisador` e use o seu ORCID.
4. **Um indicador novo.** Que % da produção está em periódicos **nacionais** abertos? *(Dica: o campo `fonte_editora` e o filtro `has_doi`.)*
5. **APC evitável.** Quanto foi pago em APC para publicar em periódicos **híbridos** — isto é, em revistas que já cobram assinatura?

### Vibe coding: peça ao LLM

Todo o código está visível acima. Copie a função que quer mudar, cole num LLM e peça. Prompts que funcionam bem:

```
Aqui está minha função montar_filtro (cole o código).
Adicione suporte a filtrar por financiador (funder) do OpenAlex,
mantendo a mesma assinatura e as mensagens de erro em português.
```

```
Tenho um DataFrame com as colunas: (cole df.dtypes).
Escreva uma função que calcule, por ano, o % de obras em periódicos
cuja fonte_editora está numa lista de editoras comerciais que eu passo
como parâmetro. Retorne um DataFrame com ano, obras, percentual.
```

**Sempre confira o que o LLM devolveu**: rode em um recorte pequeno cujo resultado você consiga verificar na mão, e compare com a contagem do próprio OpenAlex (`cliente.count('works', {'filter': ...})`).

---

### Limites destes dados — leia antes de publicar qualquer número

| Limite | O que fazer |
|---|---|
| **Vínculo institucional** é inferido pela afiliação declarada na publicação. Autor sem afiliação escrita fica de fora. | Compare o total do OpenAlex com o do seu repositório/CRIS antes de reportar. |
| **`oa_status`** vem do Unpaywall e reflete o estado **hoje**, não na data da publicação. | Não use para série histórica de "quando abriu". |
| **APC** só aparece quando o OpenAlex consegue inferir o valor. Acordos e isenções não aparecem. | Reporte sempre com a cobertura do dado. É um piso. |
| **Citações** dependem da cobertura do OpenAlex, que difere de Scopus/WoS. | Não compare números de bases diferentes. |
| Obras **sem DOI** têm metadados mais pobres. | Considere `has_doi:true` quando a qualidade importar mais que a completude. |

### Para levar

- Repositório: [`https://github.com/fabiocantoadv/OA_monitor`](https://github.com/fabiocantoadv/OA_monitor)
- Documentação da API: [docs.openalex.org](https://docs.openalex.org)
- Bracco, L. (2022). *Promoting open science through bibliometrics: A practical guide to building an open access monitor.* Liber Quarterly, 32, 1–18. [doi:10.53377/lq.11545](https://doi.org/10.53377/lq.11545)

Dados do OpenAlex sob licença **CC0**. Este material sob **CC BY 4.0**.